# Lecture 01 — What is crawling?

> *"Anyone with a Python interpreter can scrape. The skill is in scraping in a way that doesn't get you blocked, sued, or hated."*

In Lecture 00 we built the mental model of the web. Now we put a name to what we're going to do with it.

## What you'll be able to do after this lecture

- Distinguish *crawling*, *scraping*, and *parsing*, and use the words correctly.
- Sketch the lifecycle of a real-world crawler in five stages.
- Read and respect a `robots.txt` file.
- Identify the legal and ethical questions to ask *before* writing a single line of code.
- Decide, for any given site, whether you should crawl it at all.


## 1. Crawling vs. scraping vs. parsing

These three words get used interchangeably and that's fine in casual conversation. But when you're debugging — or arguing with a lawyer, or writing a job description — the distinction matters.

| Term       | What it means                                                                  | Example                                                  |
|------------|--------------------------------------------------------------------------------|----------------------------------------------------------|
| **Crawling**  | Discovering URLs and walking from page to page across a site or the web.   | Googlebot following every link from one news article to the next. |
| **Scraping**  | Pulling specific data values out of a page after you have it.              | "Get me the title and price from this product page."     |
| **Parsing**   | Turning a blob of bytes (HTML, JSON, XML) into a structured object.        | `BeautifulSoup(html, "lxml")` building a DOM tree.       |

A typical real project does all three: it **crawls** to find pages, **parses** each one, and **scrapes** the fields you care about. We will eventually build something that does all three end-to-end (lecture 04), but the words let you describe each piece precisely.


## 2. The crawler lifecycle

Almost every crawler, from a 30-line script to a search engine, has the same five stages:

```
   ┌────────────┐    ┌────────┐    ┌────────┐    ┌─────────┐    ┌────────┐
   │ Seed URLs  │ ─> │ Fetch  │ ─> │ Parse  │ ─> │ Extract │ ─> │ Store  │
   └────────────┘    └────────┘    └────────┘    └─────────┘    └────────┘
                          ▲                          │
                          │                          ▼
                          └────  Discover new URLs ──┘
```

1. **Seed URLs.** Where do you start? Often a single URL, sometimes a sitemap, sometimes a list of search queries.
2. **Fetch.** Send an HTTP request. Get bytes back. (Lecture 02.)
3. **Parse.** Turn the bytes into a tree you can query. (Lecture 03.)
4. **Extract.** Pull out the structured fields you actually want, and *also* any new URLs the crawler should follow.
5. **Store.** Persist the extracted records somewhere durable. JSONL, SQLite, S3, a real database — pick what fits the project.

The **discover-new-URLs** loop is what makes a crawler different from a one-page scraper. A crawler builds its own work queue as it runs. This is exactly how Google indexes the web (just at a different scale).

Each stage is a place where things go wrong:
- **Fetch** fails: timeout, rate limit, network blip, captcha.
- **Parse** fails: HTML is malformed, encoding wrong, structure changed.
- **Extract** fails: selector doesn't match because the page changed, or the field is missing on this particular page.
- **Store** fails: disk full, database locked, schema mismatch.

A robust crawler accepts that all of these *will* happen and stays alive when they do. We'll see retry, backoff, and resumability patterns in later lectures.


## 3. Why do people crawl?

The legitimate reasons are surprisingly diverse. A non-exhaustive list:

- **Search indexing.** Googlebot, Bingbot, DuckDuckBot. The original crawlers.
- **Price monitoring.** Tracking competitor prices, your own prices on resellers, deals on travel.
- **Research datasets.** Building corpora for NLP, ML, social-science papers.
- **News aggregation.** Pulling headlines from many outlets into one feed.
- **Archiving.** The Internet Archive's Wayback Machine is "just" a giant crawler.
- **Compliance / brand protection.** Watching for counterfeit listings, leaked credentials, copyright violations.
- **Lead generation.** (Often spammy, sometimes legitimate, usually contentious.)
- **Personal automation.** "Tell me when this concert tour adds a Seoul date."

The illegitimate reasons exist too: scraping personal data for resale, building datasets that violate privacy laws, mass-downloading copyrighted material. We'll talk about how to tell the difference in section 5.


## 4. The unwritten contract

When you point a crawler at someone else's server, you are using their resources without paying. That puts you in a relationship with them, even if they don't know your name. There are three rules that govern this relationship:

1. **Don't break their stuff.** Hammering a small site at 1000 requests/second can effectively DoS it. Slow down.
2. **Don't lie about who you are.** Use a real `User-Agent`. Many people put their email or project URL in it: `MyCrawler/0.1 (+https://github.com/me/mycrawler)`. If something goes wrong, the operator can contact you.
3. **Respect their wishes when they bother to express them.** That's what `robots.txt` is for.

Now: none of these are legally binding in most places. You *can* ignore them. People do, and many people get away with it. But:
- Violating them is the surest way to get your IP banned.
- Violating them is the surest way to make scraping illegal *for everyone else*. Every aggressive crawler raises the stakes.
- Violating them at scale, especially against named individuals' data, can land you in real legal trouble.

Be the kind of crawler operator the open web can survive. We need a few more of those.


## 5. `robots.txt` — what it is and isn't

Almost every site has a file at `/robots.txt`. It's a plain-text file describing which paths bots are *asked* not to fetch. It's a *request*, not a wall — there's no enforcement, just etiquette.

The syntax is small. Here's a typical example:

```
User-agent: *
Disallow: /admin/
Disallow: /search?
Crawl-delay: 5

User-agent: Googlebot
Allow: /search?
```

Read it as:
- For *everyone* (`*`), don't crawl `/admin/` or anything starting with `/search?`. Leave 5 seconds between requests.
- For Googlebot specifically, `/search?` is OK.

`robots.txt` is **advisory**. Your crawler still has to *choose* to obey it. Python ships with a parser:


In [ ]:
from urllib.robotparser import RobotFileParser

rp = RobotFileParser()
rp.set_url("https://en.wikipedia.org/robots.txt")
rp.read()

for path in ["/", "/wiki/Web_crawler", "/w/api.php", "/wiki/Special:Random"]:
    allowed = rp.can_fetch("MyCrawler/0.1", path)
    print(f"  {'OK ' if allowed else 'NO '}  https://en.wikipedia.org{path}")

# crawl_delay is also exposed if the file specifies one
print("crawl_delay:", rp.crawl_delay("MyCrawler/0.1"))


**What `robots.txt` does NOT do:**
- It doesn't authorize anything that would otherwise be illegal. "But robots.txt allowed it!" is not a legal defense.
- It doesn't make ignored entries safe. `Disallow:` paths often hide admin pages, internal search, or expensive endpoints — exactly the things you'd most upset the operator by hitting.
- It doesn't apply to humans. You can still browse `/admin/` in your browser if it's not behind auth. The file targets bots specifically.

**Rule of thumb:** treat `robots.txt` like a "Please use the side entrance" sign. You *could* walk in the front. You won't, because you're polite.


## 6. Terms of Service, copyright, and personal data

`robots.txt` is the cheap signal. The heavier signals are:

### Terms of Service (ToS)

Almost every site has a ToS document. Many explicitly forbid automated access. Whether ToS is contractually enforceable when you didn't click "I agree" is a real legal debate (the *browsewrap* vs *clickwrap* distinction in US law). What you can rely on:
- Sites *can* and *do* ban accounts and IPs based on ToS violations.
- Sites *can* sue, and sometimes win, especially when you cause them measurable harm.
- High-profile cases worth knowing: **hiQ Labs v. LinkedIn** (US, 2017–2022) — public profile scraping survived a CFAA challenge but the ToS-breach piece is more complicated than the headlines suggest.

### Copyright

The HTML and the data on a page may be copyrighted. Crawling for *personal use* is generally treated leniently. **Republishing** scraped content is where you get into trouble.

### Personal data

If your crawl picks up names, emails, photos, addresses — you are now processing personal data. Depending on jurisdiction:
- **EU**: GDPR applies. Lawful basis required, data-subject rights, big fines.
- **South Korea**: PIPA (Personal Information Protection Act). Strict, with extraterritorial reach.
- **California**: CCPA / CPRA.
- **Most places**: at minimum, expect anger.

If your project doesn't *need* personal data, don't collect it. If it does, talk to a lawyer before you build it.


## 7. The decision framework

Before you write a crawler, run through this checklist. If you can't answer "yes, I'm comfortable" to all of them, stop and reconsider.

```
1. Does the site have a public API or data dump? If yes → use that, you're done.
2. Does robots.txt allow the paths I want?
3. Have I read the ToS for relevant clauses about automated access?
4. Will my volume meaningfully impact their server (think: a few percent of their traffic)?
5. Am I collecting personal data? If yes, do I have a lawful basis?
6. If a person from the site emailed me asking what I'm doing, would I be comfortable explaining it?
```

Question 6 is the most useful. It's not a legal test — it's a smell test. If your gut says "I'd be embarrassed", trust that.


## 8. Identify yourself

When you're ready to crawl, your `User-Agent` should look something like:

```
NewsAggregator/0.3 (+https://example.com/aggregator/about; vova.e.125@gmail.com)
```

Three pieces:
- **Name and version** of your crawler so the operator can recognize repeat traffic.
- **A URL** explaining the project, ideally with a way to ask you to stop.
- **A contact email**.

This isn't legally required. It's just professional. Many site operators won't block a polite crawler that asks first; many *will* block one that pretends to be Chrome.

We'll set this up in code in lecture 02.


## 9. Recap

- **Crawling** discovers pages, **scraping** extracts fields, **parsing** turns bytes into trees.
- A crawler is a five-stage loop: seed → fetch → parse → extract → store, with discovery feeding back to fetch.
- `robots.txt` is a polite request, not a fence. Read it. Obey it.
- Terms of Service, copyright, and personal-data law are the heavier constraints. Take them seriously, especially when republishing or handling personal info.
- Identify your crawler. Be the operator you'd want others to be.

## Exercises

1. Pick a site you might want to scrape. Fetch its `robots.txt`. List three paths it disallows and guess *why*.
2. Read the relevant section of that site's Terms of Service (search for "automated", "robot", "crawler", or "scraper"). Summarize the policy in two sentences.
3. Apply the 6-question framework. Decide: would you crawl this site? Write 100 words explaining your reasoning. There is no right answer.
4. Sketch (no code) the five lifecycle stages for a project that builds a dataset of "all Korean indie albums released in 2025". What would the seed URLs be? What would you store?

## Up next

**Lecture 02** — we get our hands dirty. HTTP in Python: `httpx`, sessions, cookies, retries, the headers that matter. By the end you'll be able to fetch any public URL like a professional.
